In [0]:
dbutils.widgets.removeAll()

In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.text("input_table_name", input_table_list[0])
dbutils.widgets.dropdown("embedding_model", embedding_model_list[0], embedding_model_list)

catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
input_table_name=dbutils.widgets.get("input_table_name")
embedding_model=dbutils.widgets.get("embedding_model")
df = spark.read.table(f"{catalog}.{schema}.{input_table_name}")
table_name = f"{catalog}.{schema}.{input_table_name}"
print(f"table_name:{table_name}, embedding_model:{embedding_model}")


In [0]:
# Check if column exists, then add if it doesn't
try:
    spark.sql(f"DESCRIBE {table_name}").filter("col_name = 'text_embedding'").count()
    print("Column text_embedding already exists")
except:
    spark.sql(f"""
    ALTER TABLE {table_name}
    ADD COLUMNS (text_embedding ARRAY<FLOAT>)
    """)
    print("Column text_embedding added")

In [0]:
def get_ai_merge_sql(table_name, model_name, ai_result_column="text_embedding", input_column="text" ):
  sql_str=f"""  MERGE INTO {table_name} t
    USING (
      SELECT
        id,
        ai_query("{model_name}", {input_column}) AS embedding
      FROM {table_name}
      WHERE {ai_result_column} IS NULL
        AND {input_column} IS NOT NULL and {input_column}!= ''
      LIMIT 10
    ) s
    ON t.id = s.id
    WHEN MATCHED AND t.{ai_result_column} IS NULL THEN
      UPDATE SET t.{ai_result_column} = s.embedding;
  """
  return sql_str

In [0]:
sql_str=get_ai_merge_sql(table_name=table_name, model_name=embedding_model)

In [0]:
while True:
    null_count = spark.sql(f"""
        SELECT COUNT(*) AS c
        FROM {table_name}
        WHERE text_embedding IS NULL
          AND text IS NOT NULL and text <>""
    """).collect()[0]["c"]

    if null_count == 0:
        print("Done. No more NULL embedding.")
        break

    print(f"Remaining records to embed: {null_count}. Processing next batch...")
    spark.sql(sql_str)

In [0]:
#from pyspark.sql.functions import col
#spark.table("amit.default.v_comments_with_topic_toxicity").filter(col("toxicity").isNotNull()).filter(col("topic").isNotNull()).\
#select("comment_id", "message", "translated").withColumnRenamed("comment_id","id").withColumnRenamed("message", "text").dropDuplicates(["text"]).limit#(100)\
#.write.mode("overwrite").format("delta").saveAsTable("amit.bertopic.bertopic_input")